In [14]:
import pandas, numpy, geopandas, rasterio, sklearn, matplotlib, pygbif, shapely
print("all good")

all good


In [15]:
from pygbif import occurrences as occ
import pandas as pd

all_results = []
for offset in range(0, 1500, 300):
    r = occ.search(scientificName="Panthera uncia", hasCoordinate=True, limit=300, offset=offset)
    all_results.extend(r["results"])
    if r["endOfRecords"]:
        break

df = pd.DataFrame(all_results)
print(len(df), "records")

421 records


The data was downloaded from GBIF on 13 oct 2026. So there might be difference while re-running as GBIF updates it's data.

In [16]:
print(df.shape)
print(df["countryCode"].value_counts())
print(df["basisOfRecord"].value_counts())
print(df["year"].describe())


(421, 178)
countryCode
IN    106
US    105
CN     60
MN     50
UZ     41
KG     18
NP     12
RU      8
PK      6
KZ      5
TJ      3
CA      2
DE      2
AF      1
JP      1
SE      1
Name: count, dtype: int64
basisOfRecord
HUMAN_OBSERVATION      237
PRESERVED_SPECIMEN     126
MACHINE_OBSERVATION     41
MATERIAL_SAMPLE          9
MATERIAL_CITATION        8
Name: count, dtype: int64
count     406.000000
mean     2014.399015
std        21.216891
min      1896.000000
25%      2015.000000
50%      2020.000000
75%      2024.000000
max      2026.000000
Name: year, dtype: float64


In [17]:
cols = ["decimalLatitude", "decimalLongitude", "year", "countryCode",
        "stateProvince", "basisOfRecord", "coordinateUncertaintyInMeters"]
d = df[[c for c in cols if c in df.columns]].copy()

d = d.dropna(subset=["decimalLatitude", "decimalLongitude"])
d = d.drop_duplicates(subset=["decimalLatitude", "decimalLongitude", "year"])
d = d[d["year"] >= 1970]

# rough Himalaya / Central Asian mountain box (we can refine later)
d = d[d["decimalLongitude"].between(70, 105) & d["decimalLatitude"].between(25, 40)]



In [18]:
s = d[d["basisOfRecord"] == "PRESERVED_SPECIMEN"]
print(len(s))
print(s["coordinateUncertaintyInMeters"].describe())
print(s["year"].describe())
# one record is off by 1000 Kilometers and 2 obsrevation are from year 1979 and 1998 which is too old for our purposes.

2
count          1.0
mean     1000000.0
std            NaN
min      1000000.0
25%      1000000.0
50%      1000000.0
75%      1000000.0
max      1000000.0
Name: coordinateUncertaintyInMeters, dtype: float64
count       2.000000
mean     1988.500000
std        13.435029
min      1979.000000
25%      1983.750000
50%      1988.500000
75%      1993.250000
max      1998.000000
Name: year, dtype: float64


In [19]:
d = d[d["basisOfRecord"].isin(["HUMAN_OBSERVATION", "MACHINE_OBSERVATION"])]
print(len(d), "records after keeping only observations")
s = d[d["basisOfRecord"] == "PRESERVED_SPECIMEN"]
print(len(s))
print(s["coordinateUncertaintyInMeters"].describe())
print(s["year"].describe())
# First Pass 165 records came when we filtered out the preserved specimens, so we have 165 records that are human or machine observations.
#There are no unceratinity filters in this dataset so these records are not final records.

165 records after keeping only observations
0
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: coordinateUncertaintyInMeters, dtype: float64
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: year, dtype: float64


In [20]:
print(d["coordinateUncertaintyInMeters"].describe())
print((d["coordinateUncertaintyInMeters"] > 5000).sum(), "records with uncertainty over 5 km")
# When we are taking unceratinity over 5 km, 160 records out of 162 are over 5 km so we are left with almost nothing.

count       162.000000
mean      63217.067901
std       78257.774983
min           5.000000
25%       28874.000000
50%       29066.000000
75%       65192.000000
max      515959.000000
Name: coordinateUncertaintyInMeters, dtype: float64
160 records with uncertainty over 5 km


In [21]:
print(df["datasetName"].value_counts().head())
print(df["institutionCode"].value_counts().head())
# This tells us that 227 of our records come from the inaturalist research-grade observations and because of that most of our 
# data is off by 29 km so  I used a coarser 10 arc-minute (about 18 km) grid".

datasetName
iNaturalist research-grade observations    227
Mammalia (Luomus)                            2
KUO Mammal collections                       1
NMNH Extant Biology                          1
Utenlandsk samling (Pattedyr)                1
Name: count, dtype: int64
institutionCode
iNaturalist                               227
MSB                                        79
Chatkal State Biosphere Nature Reserve     38
UWBM                                       11
ZFMK                                        8
Name: count, dtype: int64


In [22]:
cols = ["decimalLatitude", "decimalLongitude", "year", "countryCode",
        "basisOfRecord", "coordinateUncertaintyInMeters"]
d = df[[c for c in cols if c in df.columns]].copy()

d = d.dropna(subset=["decimalLatitude", "decimalLongitude"])
d = d.drop_duplicates(subset=["decimalLatitude", "decimalLongitude", "year"])
d = d[d["year"] >= 1970]

# wider High Mountain Asia box (no country filter)
d = d[d["decimalLongitude"].between(65, 110) & d["decimalLatitude"].between(25, 50)]

d = d[d["basisOfRecord"].isin(["HUMAN_OBSERVATION", "MACHINE_OBSERVATION"])]
d = d[d["coordinateUncertaintyInMeters"].isna() | (d["coordinateUncertaintyInMeters"] <= 50000)]
print(len(d), "records after final cleaning")

res = 1/6
d["cell_x"] = d["decimalLongitude"] // res
d["cell_y"] = d["decimalLatitude"] // res
d_thin = d.drop_duplicates(subset=["cell_x", "cell_y"])
print(len(d), "->", len(d_thin), "records after thinning")

print(d_thin["countryCode"].value_counts())
print(d_thin["coordinateUncertaintyInMeters"].max())

196 records after final cleaning
196 -> 102 records after thinning
countryCode
CN    28
IN    24
MN    15
KG    10
NP     8
RU     5
UZ     4
KZ     3
TJ     2
PK     2
AF     1
Name: count, dtype: int64
48572.0


We started with 421 GBIF records, removed missing coordinates, duplicates, records before 1970 and preserved specimens, kept the High Mountain Asia box (65-110°E, 25-50°N), and dropped records with uncertainty over 50 km. This left us with 196 records. Then thinning to one record per 10 arc-minute (about 18 km) cell left us with 102 records.

In [23]:
import os
os.makedirs("data", exist_ok=True)
d_thin.to_csv("data/occurrences_thinned.csv", index=False)
print("saved")

saved
